# 1) Multi-Format Delimited File Processing Pipeline

## Technical Documentation

---

## 1. Overview

### Purpose
A production-grade batch processing pipeline designed for Microsoft Fabric that validates schema consistency across multiple delimited text files. The system implements a dual-parser validation strategy combined with Spark distributed processing to handle files of any size, format, encoding, or quoting convention.

### Architecture
The pipeline employs a **staged processing architecture** with three distinct phases:

```
┌─────────────────────────────────────────────────────────────────┐
│                     PHASE 1: FILE PROFILING                     │
│  (Python open() - lightweight sampling, Fabric-compatible)      │
│  • Encoding detection (chardet)                                 │
│  • BOM detection and stripping                                  │
│  • Line ending detection                                        │
│  • Separator detection (3-layer)                                │
│  • CSV category classification (A-E)                            │
│  • Header validation and sanitization                           │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                   PHASE 2: DISTRIBUTED PROCESSING               │
│  (Spark RDDs - parallel execution across cluster)               │
│  • Text file reading via Spark textFile()                       │
│  • Dual-parser line validation on executors                     │
│  • Good/bad record classification                               │
│  • Metadata enrichment                                          │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                     PHASE 3: DELTA STORAGE                      │
│  (Spark DataFrames - ACID-compliant writes)                     │
│  • Good records → ODS.[filename]_consistent                     │
│  • Bad records → ODS.[filename]_inconsistent (only if errors)   │
│  • Comprehensive multi-file summary reporting                   │
└─────────────────────────────────────────────────────────────────┘
```

### Platform Requirements
- **Platform**: Microsoft Fabric
- **Python Environment**: PySpark environment
- **Dependencies**: `chardet` library for encoding detection
- **Lakehouse Setup**: 
  - Created a Lakehouse for raw and cleaned data storage
  - Created 2 schemas in the Lakehouse's Tables section:
    - **ODS** (Operational Data Store) - for raw delta tables, where good and bad records are stored separately, then combined before starting the cleaning process
    - **STG** (Staging) - for cleaned and data quality checked delta tables
- **File Access**: Raw data files are manually uploaded to the Lakehouse's Files section `/lakehouse/default/Files/`

---

## 2. RFC 4180 and Delimited File Standards

### The RFC 4180 Standard

RFC 4180 is the Internet Engineering Task Force (IETF) specification that defines the Common Format and MIME Type for Comma-Separated Values (CSV) Files. Published in October 2005, it provides the formal grammar for how CSV files should be structured and parsed.

**Core Rules Defined by RFC 4180:**

| Rule | Specification | Example |
|------|---------------|---------|
| **Field Separation** | Fields are separated by commas | `John,25,NYC` |
| **Quoted Fields** | Fields containing commas, line breaks, or double quotes must be enclosed in double quotes | `"123 Main St, Apt 4"` |
| **Escaped Quotes** | Double quotes inside quoted fields are escaped by doubling them | `"He said ""Hello"""` |
| **Line Breaks** | Rows are separated by CRLF; the last record may omit the trailing line break | `\r\n` |
| **Optional Header** | The first row may contain column names; presence is not mandated | Header row followed by data rows |
| **Consistent Columns** | Each row must have the same number of fields throughout the file | All rows match header count |
| **MIME Type** | Registered as `text/csv` | Content-Type header |

### RFC 4180 Across Different Delimiters

While RFC 4180 explicitly defines the comma-separated format, its **quoting and escaping rules are universally applied** to other delimited formats through convention and parser implementations:

| Format | Delimiter | RFC 4180 Quoting Rules Apply? | Standard Body |
|--------|-----------|-------------------------------|---------------|
| **CSV** | Comma (`,`) | ✅ Yes - RFC 4180 defines this format | IETF RFC 4180 |
| **TSV** | Tab (`\t`) | ✅ Yes - IANA registered as `text/tab-separated-values` | IANA Media Types |
| **Pipe-Delimited** | Pipe (`\|`) | ✅ Convention - Follows RFC 4180 quoting semantics | No formal RFC; industry convention |
| **Semicolon-Delimited** | Semicolon (`;`) | ✅ Convention - Common in European locales where comma is decimal separator | No formal RFC; regional convention |

**Key Insight**: The quoting and escaping mechanism defined in RFC 4180 (double quotes around fields containing special characters, doubled quotes for escaping) is a **general-purpose text delimiting strategy** that applies regardless of the delimiter character. All `csv.reader` implementations apply these rules to any specified delimiter.

### ISO Standards for Delimited Files

While there is no single ISO standard that mirrors RFC 4180 exactly, several related ISO standards apply:

| Standard | Relevance |
|----------|-----------|
| **ISO/IEC 8859** | Character encoding standards for delimited files in various languages |
| **ISO 8601** | Date/time format standard; applies to date columns in delimited files |
| **ISO/IEC 10646** | Universal Coded Character Set (UCS/Unicode); defines UTF-8/UTF-16 encoding |
| **ISO 8000** | Data quality standards; framework for measuring and improving data quality |
| **ISO 9735** | EDIFACT standard for electronic data interchange; uses delimited segments |

**What This Pipeline Implements**: The pipeline follows RFC 4180 quoting rules for all delimiter types (comma, tab, pipe, semicolon), applies ISO 8601 for timestamp formatting, and detects ISO/IEC 8859 and ISO/IEC 10646 encodings via chardet.

---

## 3. Dual-Parser Validation Strategy

### Architecture

Every row in the input file is simultaneously parsed by two independent engines:

```
                         ┌──────────────────────┐
                         │   Raw Delimited Line │
                         └──────────┬───────────┘
                                    │
                   ┌────────────────┴────────────────┐
                   │                                 │
                   ▼                                 ▼
         ┌──────────────────┐              ┌──────────────────┐
         │   csv.reader     │              │   str.split()    │
         │  (RFC 4180)      │              │ (Simple Split)   │
         │  Quote-Aware     │              │ Quote-Unaware    │
         └────────┬─────────┘              └────────┬─────────┘
                  │                                 │
                  ▼                                 ▼
         ┌──────────────────┐              ┌──────────────────┐
         │  fields_csv[]    │              │  fields_split[]  │
         └────────┬─────────┘              └────────┬─────────┘
                  │                                 │
                  └──────────────┬──────────────────┘
                                 │
                                 ▼
                   ┌──────────────────────────┐
                   │  Compare Field Counts     │
                   │  Against Expected Schema  │
                   └──────────────┬───────────┘
                                  │
                   ┌──────────────┴──────────────┐
                   │                             │
              Counts Agree                  Counts Differ
                   │                             │
                   ▼                             ▼
         ┌──────────────────┐        ┌────────────────────────┐
         │ Use csv.reader   │        │ Flag: PARSER_DISAGREE  │
         │ Both parsers     │        │ Select parser matching │
         │ in agreement     │        │ expected column count  │
         └──────────────────┘        └────────────────────────┘
```

### Decision Matrix

| csv.reader Fields | split() Fields | Matches Expected? | Row Classification | Parser Disagreement |
|-------------------|----------------|-------------------|-------------------|---------------------|
| = Expected | = Expected | Both match | **GOOD** | NO |
| = Expected | ≠ Expected | csv.reader matches | **GOOD** | YES |
| ≠ Expected | = Expected | split matches | **GOOD** (fallback) | YES |
| ≠ Expected | ≠ Expected | Neither matches | **BAD** | NO |

### Parser Disagreement Significance

When the parsers produce different field counts, it indicates one of two scenarios:

| Scenario | csv.reader | split() | Root Cause |
|----------|-----------|---------|------------|
| **Quoted Fields with Embedded Delimiters** | Correct count | Incorrect count (higher) | File contains RFC 4180-compliant quoted fields; split() fragments them |
| **Malformed/Broken Quoting** | Incorrect count | Correct count | Unbalanced quotes confuse the state-machine parser; split() ignores quote context |

### Error Types Generated

| Error Type | Condition | Typical Cause |
|------------|-----------|---------------|
| `MISSING_COLUMNS` | Actual fields < expected | Truncated rows, incomplete data export |
| `EXTRA_COLUMNS` | Actual fields > expected | Unquoted delimiters in data, schema mismatch |
| `PARSE_ERROR` | csv.reader throws exception | Completely malformed line, binary data, encoding corruption |

---

## 4. CSV Quoting Categories and Format Variations

The pipeline classifies every file into one of five categories based on its quoting behavior. This classification applies to **all delimiter types** (comma, tab, pipe, semicolon).

### Category A: Unquoted (Clean)

**Characteristics**:
- No fields wrapped in double quotes
- No delimiter characters present in field values
- Both parsers produce identical results for all rows

**Example (Comma-Delimited)**:
```
Name,Age,City
John,25,New York
Jane,30,Los Angeles
Bob,35,Chicago
```

**Parser Behavior**:
| Parser | Row 1 (John) | Row 2 (Jane) | Row 3 (Bob) |
|--------|-------------|-------------|-------------|
| csv.reader | 3 fields ✅ | 3 fields ✅ | 3 fields ✅ |
| split() | 3 fields ✅ | 3 fields ✅ | 3 fields ✅ |
| Agreement | YES | YES | YES |

**Example (Tab-Delimited)**:
```
Name	Age	City
John	25	New York
Jane	30	Los Angeles
```

**Prevalence**: Common in machine-generated exports where field values are guaranteed to not contain the delimiter.

---

### Category B: Fully Quoted

**Characteristics**:
- Every field in every row is wrapped in double quotes
- Delimiter characters may exist inside quoted fields
- Both parsers may produce different counts if embedded delimiters exist inside quotes

**Example (Comma-Delimited)**:
```
"Name","Address","City"
"John","123 Main St, Apt 4","New York"
"Jane","456 Oak Ave, Unit 2","Los Angeles"
```

**Parser Behavior**:
| Parser | Row 1 (John) | Row 2 (Jane) |
|--------|-------------|-------------|
| csv.reader | 3 fields ✅ | 3 fields ✅ |
| split() | 5 fields ❌ | 4 fields ❌ |
| Agreement | NO (Row 1, Row 2) | NO |

**Example (Pipe-Delimited)**:
```
"ID"|"Description"|"Status"
"1"|"This needs | urgent | attention"|"open"
"2"|"Normal task"|"closed"
```

**Parser Behavior**:
| Parser | Row 1 | Row 2 |
|--------|-------|-------|
| csv.reader | 3 fields ✅ | 3 fields ✅ |
| split("\|") | 5 fields ❌ | 3 fields ✅ |
| Agreement | NO | YES |

**Prevalence**: Common in ETL exports, database dumps, and systems that quote all fields defensively.

---

### Category C: Partially Quoted

**Characteristics**:
- Only fields containing delimiter characters are wrapped in double quotes
- Unquoted fields and quoted fields coexist in the same row
- Parser disagreements occur only on rows containing quoted fields

**Example (Comma-Delimited)**:
```
Name,Address,City,Zip
John,"123 Main St, Apt 4",New York,10001
Jane,456 Oak Ave,Los Angeles,90001
Bob,"789 Pine Rd, Suite 7",Chicago,60601
```

**Parser Behavior**:
| Parser | Row 1 (John) | Row 2 (Jane) | Row 3 (Bob) |
|--------|-------------|-------------|-------------|
| csv.reader | 4 fields ✅ | 4 fields ✅ | 4 fields ✅ |
| split() | 5 fields ❌ | 4 fields ✅ | 5 fields ❌ |
| Agreement | NO | YES | NO |

**Example (Tab-Delimited)**:
```
Name	Address	City
John	"123 Main St	Apt 4"	New York
Jane	456 Oak Ave	Los Angeles
```

**Prevalence**: Most common format in production environments. Applications like Excel and Google Sheets produce this format when exporting data containing commas in cells.

---

### Category D: Unquoted (Ambiguous)

**Characteristics**:
- No quoting present in the file
- Delimiter characters appear within field values
- Both parsers produce identical but incorrect counts
- Represents a **genuine schema error** or **malformed data export**

**Example (Comma-Delimited)**:
```
Name,Address,City
John,123 Main St, Apt 4,New York
Jane,456 Oak Ave,Los Angeles
```

**Parser Behavior**:
| Parser | Row 1 (John) | Row 2 (Jane) |
|--------|-------------|-------------|
| csv.reader | 4 fields ❌ | 3 fields ✅ |
| split() | 4 fields ❌ | 3 fields ✅ |
| Agreement | YES (both wrong) | YES (both correct) |

**Key Insight**: Because both parsers agree, this cannot be distinguished from a genuine extra column. Downstream recovery must decide whether to merge the fragmented fields or flag as an error.

**Prevalence**: Occurs when applications export data without proper CSV quoting, or when raw data is manually edited without understanding quoting requirements.

---

### Category E: Mixed/Malformed

**Characteristics**:
- Inconsistent or broken quoting patterns
- Unbalanced quotes, unescaped internal quotes
- csv.reader may fail or produce unexpected results
- split() may produce correct results by ignoring quote context entirely

**Example**:
```
Name,Description,Price
John,"Broken "quote inside,10
Jane,Normal text,20
Bob,"Missing end quote,30
```

**Parser Behavior**:
| Parser | Row 1 (John) | Row 2 (Jane) | Row 3 (Bob) |
|--------|-------------|-------------|-------------|
| csv.reader | 4 fields ❌ | 3 fields ✅ | 2 fields ❌ (unclosed quote) |
| split() | 3 fields ✅ | 3 fields ✅ | 3 fields ✅ |
| Agreement | NO | YES | NO |

**Prevalence**: Common in manually-edited files, concatenated exports, or systems with buggy CSV generators.

---

## 5. File Property Detection System

### Encoding Detection

The pipeline automatically detects the character encoding of each file before processing. This is critical because:

- **UTF-8** files may contain multi-byte characters (emoji, CJK characters, accented letters)
- **UTF-16** files use 2 or 4 bytes per character and cannot be read as UTF-8
- **Latin-1/ISO-8859-1** files use single-byte encoding common in Western European systems
- Reading with the wrong encoding produces garbled text or `UnicodeDecodeError`

**Implementation**: The `chardet` library analyzes raw bytes from the beginning of the file and returns the most likely encoding with a confidence score (0.0 to 1.0). Confidence below 70% generates a warning.

### BOM (Byte Order Mark) Handling

A BOM is a special character sequence at the beginning of a file that indicates encoding:

| BOM Bytes | Encoding |
|-----------|----------|
| `EF BB BF` | UTF-8 |
| `FF FE` | UTF-16 Little Endian |
| `FE FF` | UTF-16 Big Endian |

**Problem**: If BOM is not stripped, the first column header becomes corrupted:
```
Before stripping: "\ufeffID" → sanitized to "_id" or similar
After stripping:  "ID" → sanitized correctly to "id"
```

**Solution**: The pipeline detects BOM presence and reads from the byte after the BOM marker.

### Line Ending Detection

Different operating systems use different line ending conventions:

| Style | Characters | Escape Sequence | Common Source |
|-------|-----------|-----------------|---------------|
| Unix/Linux | Line Feed | `\n` | Linux, macOS 10+, modern systems |
| Windows | Carriage Return + Line Feed | `\r\n` | Windows, DOS, legacy systems |
| Old Mac | Carriage Return | `\r` | Mac OS 9 and earlier |

**Impact on Processing**:
- Incorrect line ending handling can cause lines to be merged or split incorrectly
- Multi-line quoted fields interact with line ending detection
- Line counting for error reporting depends on correct line ending identification

**Implementation**: Counts occurrences of each line ending style in the sample bytes and selects the most frequent.

### Binary File Detection

**Problem**: Files with CSV extensions may actually contain binary data (Excel files, PDFs, images saved with wrong extension).

**Detection**: If null bytes (`\x00`) constitute more than 10% of the sample, the file is flagged as likely binary.

**Handling**: A warning is displayed, and processing continues. Binary content will produce `PARSE_ERROR` rows that can be reviewed downstream.

---

## 6. Separator Detection System

The pipeline implements a **three-layer separator detection system** designed to handle edge cases that defeat simpler heuristics.

### Layer 1: csv.Sniffer with Full Dialect Detection

`csv.Sniffer` is Python's built-in CSV dialect detector. It analyzes:
- Delimiter character
- Quote character
- Escape character
- Line ending convention

Unlike simple field counting, `Sniffer` understands CSV grammar. It knows that delimiters inside quotes are not structural separators.

**Example where Sniffer succeeds**:
```
name	address	city
John	123 Main St, Apt 4	New York
```
Sniffer correctly identifies `\t` as the delimiter because it recognizes that commas appear inside fields but tabs are consistently structural.

**Validation**: After Sniffer makes its choice, the pipeline validates by parsing sample lines and checking consistency. If validation fails (consistency < 80%), Layer 2 is invoked.

### Layer 2: Quote-Aware Consistency Scoring with Content Validation

This layer evaluates each candidate delimiter by three metrics:

1. **Consistency Score** (50% weight): How consistently does the delimiter produce the same field count across sample lines?
2. **Non-Empty Ratio** (20% weight): What proportion of parsed fields contain actual content?
3. **Variation Score** (30% weight): Do the parsed fields exhibit natural length variation (real data) or identical lengths (likely wrong delimiter)?

The composite score is multiplied by the most common field count to favor structured data (more columns typically indicates a real delimiter).

**Example where Layer 2 outperforms simple counting**:
```
Single-column file with commas:
notes
"Purchased apples, oranges, and bananas"
"Meeting at 3pm, don't be late"
```
- Comma counts: [2, 2] — consistent but wrong
- Tab counts: [0, 0] — consistent but wrong
- Layer 2: Both produce field count 1 consistently; non-empty ratio and variation select the delimiter that gives most meaningful fields

### Layer 3: Single-Column Fallback

If all delimiters produce identical results (single-column file), the system defaults to tab as a neutral choice. The file will be processed as a single-column dataset.

---

## 7. Quotation Field Detection

### The False Positive Problem

Simple substring matching for quote detection produces false positives:

```python
field = "123"
line = '"abc123def","other","fields"'
# Substring check: '"123"' in line → True (FALSE POSITIVE)
# The "123" matched inside "abc123def" which is a different field
```

### The Regex Solution

The pipeline uses regex with escaped field values to ensure only exact matches:

```python
escaped_field = re.escape(field)
pattern = f'"{escaped_field}"'
# re.search(pattern, line) → None (CORRECT - no standalone "123" field)
```

This ensures that field `"123"` only matches when it appears as a complete quoted field, not as a substring within another field.

---

## 8. Header Validation System

### Validation Checks Performed

| Check | Condition | Action |
|-------|-----------|--------|
| **Empty File** | First line is empty or file has zero bytes | Return error; skip processing |
| **Empty Header** | First line parsed but produces zero fields | Return error; skip processing |
| **Empty Column Names** | Individual column name is empty string or whitespace | Rename to `column_N` where N is position |
| **Duplicate Column Names** | Two or more columns have identical names after sanitization | Suffix duplicates with `_2`, `_3`, etc. |
| **BOM Corruption** | First column name starts with BOM character | Strip BOM before parsing |

### Column Name Sanitization Rules

| Input | Operation | Output |
|-------|-----------|--------|
| `"First Name"` | Remove spaces and special chars | `first_name` |
| `"123Data"` | Prefix with underscore if starts with digit | `_123data` |
| `""` (empty) | Replace with placeholder | `column_1` |
| `"Name"` and `"name"` | Lowercase and deduplicate | `name`, `name_2` |
| `"Col@#1"` | Remove special characters | `col_1` |

---

## 9. Spark Distributed Processing Architecture

### Problem Solved: Driver Memory Bottleneck

Traditional approach loads entire files into Python lists on the driver:

```python
# ANTI-PATTERN: Driver memory scales with file size
good_rows = []  # 50GB file → 50GB driver memory → OOM crash
for line in file:
    good_rows.append(parsed_line)
```

### Solution: RDD-Based Distributed Processing

The pipeline uses Spark's Resilient Distributed Datasets (RDDs) to distribute work across the cluster:

```
┌──────────────────────────────────────────────────────────────┐
│                        DRIVER NODE                           │
│  • Coordinates execution                                      │
│  • Holds schema definitions                                   │
│  • Performs final DataFrame writes                            │
│  • Memory usage: CONSTANT (does not scale with file size)     │
└──────────────────────┬───────────────────────────────────────┘
                       │
         ┌─────────────┴─────────────┐
         │                           │
         ▼                           ▼
┌──────────────────┐        ┌──────────────────┐
│   EXECUTOR 1     │        │   EXECUTOR 2     │
│  • Reads partition│        │  • Reads partition│
│  • Parses lines   │        │  • Parses lines   │
│  • Classifies     │        │  • Classifies     │
│  good/bad         │        │  good/bad         │
│  • Returns results│        │  • Returns results│
└──────────────────┘        └──────────────────┘
         │                           │
         └─────────────┬─────────────┘
                       │
         ┌─────────────┴─────────────┐
         │    (N more executors)     │
         │    Each handles its own   │
         │    partition of the file  │
         └───────────────────────────┘
```

### File Access Strategy

Spark's native CSV reader cannot directly access Fabric-mounted paths. The pipeline uses:

```python
local_path = f"file://{source_path}"
text_rdd = spark.sparkContext.textFile(local_path)
```

This reads the file through the local FUSE mount, which Fabric provides for Python file access. The `textFile()` method distributes lines across partitions automatically.

### Executor-Side Parsing

The `parse_line_worker()` function runs on executors, not the driver:

```python
def parse_line_worker(line: str, separator: str, expected_columns: int) -> Tuple:
    """Executes on Spark executors in parallel."""
    try:
        fields_csv = next(csv.reader(io.StringIO(line), delimiter=separator))
    except Exception:
        return (None, ("PARSE_ERROR", "NO", 0, 0, line[:500]))
    
    if len(fields_csv) == expected_columns:
        return (fields_csv, None)
    else:
        fields_split = line.split(separator)
        if len(fields_split) == expected_columns:
            return (fields_split, None)
        else:
            error_type = "MISSING_COLUMNS" if len(fields_csv) < expected_columns else "EXTRA_COLUMNS"
            has_quotes = any(is_quoted_field(f, line) for f in fields_csv)
            error_info = (error_type, 'YES' if has_quotes else 'NO', len(fields_csv), len(fields_split), line[:500])
            return (None, error_info)
```

### Memory Characteristics

| Component | Location | Memory Scaling |
|-----------|----------|----------------|
| File reading | Spark executors (distributed) | Per-partition, constant |
| CSV parsing | Spark executors (distributed) | Per-line, constant |
| Good/bad classification | Spark executors (distributed) | Per-line, constant |
| Schema definitions | Driver | Constant (small) |
| DataFrame writes | Driver coordinates; executors write | Constant |

---

## 10. Error Table Schema and Error Management

### Error Table Structure

When errors are detected, a `_inconsistent` Delta table is created with the following schema:

| Column | Type | Purpose |
|--------|------|---------|
| `error_type` | String | Classification: `MISSING_COLUMNS`, `EXTRA_COLUMNS`, `PARSE_ERROR` |
| `parser_disagreement` | String | `YES` if csv.reader and split() disagree; `NO` if they agree |
| `actual_fields_csv` | Integer | Field count produced by csv.reader |
| `actual_fields_split` | Integer | Field count produced by str.split() |
| `original_line` | String | Raw line content (truncated to 500 characters) |
| `load_timestamp` | String | UTC timestamp of processing |
| `source_location` | String | Original file path |
| `file_type` | String | File format identifier |

### Error Diagnostic Scenarios

| error_type | parser_disagreement | Interpretation | Recovery Strategy |
|------------|---------------------|----------------|-------------------|
| `EXTRA_COLUMNS` | NO | Genuine extra column; both parsers agree | Truncate or investigate source |
| `EXTRA_COLUMNS` | YES | Quoted field with embedded delimiters; csv.reader handled correctly | Accept csv.reader output |
| `MISSING_COLUMNS` | NO | Genuine missing column; both parsers agree | Pad with NULLs or investigate source |
| `MISSING_COLUMNS` | YES | Malformed quoting; split() resolved correctly | Accept split() output |
| `PARSE_ERROR` | NO | Completely malformed line | Human review required |

### Staged Error Recovery Architecture

The pipeline intentionally uses **binary good/bad classification** during ingestion. Error recovery is deferred to a separate downstream stage:

```
┌─────────────────────────────────────────────────────────────────┐
│                    INGESTION STAGE (This Pipeline)               │
│  • Capture ALL data                                              │
│  • Classify as good or bad                                       │
│  • Preserve original line content                                │
│  • Attach parser metadata for diagnostics                        │
│  • Write to _consistent and _inconsistent tables                 │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                    RECOVERY STAGE (Downstream)                   │
│  • Read _inconsistent tables                                     │
│  • Apply business-specific recovery rules:                       │
│    - MISSING_COLUMNS → pad with NULLs                            │
│    - EXTRA_COLUMNS + NO disagreement → flag for review           │
│    - EXTRA_COLUMNS + YES disagreement → accept csv.reader        │
│    - PARSE_ERROR → dead letter table for manual review           │
│  • Merge recovered rows into _consistent                         │
│  • Track recovery statistics                                     │
└─────────────────────────────────────────────────────────────────┘
```

**Benefits of staged recovery**:
- Recovery logic can evolve without reprocessing raw files
- Error patterns can be analyzed before automated recovery is applied
- Recovery strategies can be tested on real error data
- Rollback is possible by reprocessing from error tables

---

## 11. Delta Table Naming Convention

### Table Name Format

```
ODS.[sanitized_filename]_[suffix]

Where:
  ODS = Operational Data Store schema
  sanitized_filename = Cleaned file name (lowercase, underscores only)
  suffix = "consistent" or "inconsistent"
```

### Sanitization Rules

| Input File Name | Sanitized Name | Consistent Table | Inconsistent Table |
|-----------------|----------------|-------------------|-------------------|
| `accidents_2017.csv` | `accidents_2017` | `ODS.accidents_2017_consistent` | `ODS.accidents_2017_inconsistent` |
| `My Data (2024).csv` | `my_data_2024` | `ODS.my_data_2024_consistent` | `ODS.my_data_2024_inconsistent` |
| `123Data.csv` | `t_123data` | `ODS.t_123data_consistent` | `ODS.t_123data_inconsistent` |
| `Air Quality Nov2017.csv` | `air_quality_nov2017` | `ODS.air_quality_nov2017_consistent` | `ODS.air_quality_nov2017_inconsistent` |

### Conditional Table Creation

Error tables are only created when errors exist:
- **Clean files** (0 errors): Only `_consistent` table created
- **Dirty files** (>0 errors): Both `_consistent` and `_inconsistent` tables created
- **Failed files** (processing error): No tables created

---

## 12. Metadata Enrichment

Every processed row is enriched with metadata for complete data lineage:

| Metadata Column | Value | Purpose |
|-----------------|-------|---------|
| `load_timestamp` | UTC timestamp in `YYYY-MM-DD HH:MM:SS` format | Tracks when data was ingested |
| `source_location` | Full path in lakehouse | Traces data back to original file |
| `file_type` | File format identifier | Distinguishes CSV, TSV, pipe-delimited, etc. |

---

## 13. Processing Workflow

### Per-File Processing Sequence

```
┌─────────────────────────────────────────────────────────────────┐
│  STEP 1: FILE PROPERTY DETECTION                                 │
│  • Detect encoding (chardet)                                     │
│  • Detect and strip BOM                                          │
│  • Detect line endings                                           │
│  • Detect binary files                                           │
│  • Estimate row count                                            │
│  • Log all properties with warnings                              │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 2: SEPARATOR DETECTION                                     │
│  • Layer 1: csv.Sniffer with validation                          │
│  • Layer 2: Quote-aware consistency + content validation         │
│  • Layer 3: Single-column fallback                               │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 3: CSV CATEGORY CLASSIFICATION                              │
│  • Read sample with detected encoding                            │
│  • Parse with csv.reader and split()                             │
│  • Classify quoting patterns                                     │
│  • Assign category A/B/C/D/E                                     │
│  • Log category statistics                                       │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 4: HEADER VALIDATION                                       │
│  • Read first line with BOM handling                             │
│  • Parse header with csv.reader                                  │
│  • Detect empty column names → rename                            │
│  • Detect duplicate names → suffix                               │
│  • Sanitize all names for Delta compatibility                    │
│  • Log header warnings                                           │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 5: SPARK DISTRIBUTED PROCESSING                            │
│  • Read file as text RDD via file:// protocol                    │
│  • Strip header line                                             │
│  • Distribute lines to executors                                 │
│  • Parse each line with dual-parser worker                       │
│  • Classify as good or bad                                       │
│  • Add metadata to all records                                   │
│  • Convert to DataFrames                                         │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 6: DELTA TABLE WRITES                                      │
│  • Write good records → ODS.[file]_consistent (overwrite)         │
│  • Write bad records → ODS.[file]_inconsistent (only if errors)  │
│  • Log table creation with record counts                         │
└─────────────────────────────────────────────────────────────────┘
```

### Batch Processing Flow

```
┌─────────────────────────────────────────────────────────────────┐
│  MAIN PIPELINE LOOP                                              │
│                                                                  │
│  For each file in configured list:                               │
│    ┌─────────────────────────────────────────────────────────┐  │
│    │ 1. Execute per-file processing sequence (Steps 1-6)      │  │
│    │ 2. Collect results (counts, errors, categories)          │  │
│    │ 3. Aggregate into global statistics                      │  │
│    │ 4. If file fails → log error, continue to next file      │  │
│    └─────────────────────────────────────────────────────────┘  │
│                                                                  │
│  After all files processed:                                      │
│    ┌─────────────────────────────────────────────────────────┐  │
│    │ • Generate comprehensive multi-file summary               │  │
│    │ • Print clean/dirty/failed file breakdown                 │  │
│    │ • Print category distribution                             │  │
│    │ • Print separator breakdown                               │  │
│    │ • Print global error type aggregation                     │  │
│    │ • Print per-file summary table                            │  │
│    └─────────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

---

## 14. Summary Reporting

### Per-File Diagnostics

Each file produces:
- Detected separator and encoding
- File category (A-E) with quoting statistics
- Header column count and column names
- Parser statistics (csv.reader used, split fallback used, disagreements detected)
- Good/bad/total record counts
- Delta tables created
- Quoted file warnings when embedded delimiters detected

### Multi-File Summary

The pipeline produces:
- Total files processed with clean/dirty/failed breakdown
- Total records across all files with good/bad breakdown
- Overall success rate
- Files with parser disagreements (requires quoting-aware downstream handling)
- Separator distribution across files
- File category distribution
- Global error type aggregation

---

## 15. Error Scenarios and Management

### Scenario: File Contains Binary Data

**Detection**: `detect_file_properties()` identifies null byte concentration > 10%.

**Symptom**: Warning logged during profiling; most lines produce `PARSE_ERROR`.

**Pipeline Behavior**: Processing continues; bad rows captured in `_inconsistent` table with `PARSE_ERROR` type. Original binary content preserved for investigation.

### Scenario: Encoding Mismatch

**Detection**: `chardet` may detect wrong encoding or low confidence.

**Symptom**: Garbled characters in parsed fields; unexpected `PARSE_ERROR` rows.

**Pipeline Behavior**: Encoding logged with confidence score. Low confidence (< 70%) generates explicit warning. Operators can override encoding in configuration.

### Scenario: Large File Processing

**Detection**: Estimated row count displayed during profiling.

**Symptom**: Longer processing time but no memory issues.

**Pipeline Behavior**: Spark distributes partitions across executors. RDD caching improves performance. Memory usage remains constant regardless of file size.

### Scenario: All Rows Are Errors

**Detection**: Good count = 0, bad count = total count.

**Symptom**: Only `_inconsistent` table created; no `_consistent` table.

**Pipeline Behavior**: Error table created with all rows preserved. Operators can investigate root cause (wrong separator, wrong schema, corrupted file).

---

## 16. Configuration

| Parameter | Default | Purpose |
|-----------|---------|---------|
| `bronze_schema` | `"ODS"` | Target schema for Delta tables |
| `MAX_SAMPLE_SIZE` | `100` | Lines to sample for profiling |
| `SAMPLE_BYTES` | `8192` | Bytes to read for encoding detection |
| `file_paths` | List of 17 paths | Files to process |

---

## 17. Dependencies

| Library | Purpose |
|---------|---------|
| `chardet` | Encoding detection from raw bytes |
| `csv` (stdlib) | RFC 4180 compliant CSV parsing |
| `re` (stdlib) | Regex-based quote detection and name sanitization |
| `io` (stdlib) | In-memory string buffer for csv.reader |
| `pyspark.sql` | DataFrame operations and Delta table writes |
| `pyspark.sql.types` | Schema definitions for DataFrames |

**Installation**: `%pip install chardet` in Fabric notebook before running.


# 2) RFC 4180: The CSV Standard

### What is RFC 4180?

RFC 4180 is the Internet Engineering Task Force (IETF) standard that defines the Common Format and MIME Type for Comma-Separated Values (CSV) Files. Published in October 2005, it provides a formal specification for how CSV files should be structured and parsed, addressing ambiguities that arise from the informal nature of the format.

### Key Rules Defined by RFC 4180

| Rule | Specification | Example |
|------|---------------|---------|
| **Field Separation** | Fields are separated by commas | `John,25,NYC` |
| **Quoted Fields** | Fields containing commas, line breaks, or double quotes must be enclosed in double quotes | `"123 Main St, Apt 4"` |
| **Escaped Quotes** | Double quotes inside quoted fields are escaped by doubling them | `"He said ""Hello"""` |
| **Line Breaks** | Rows are separated by CRLF (Windows) or LF (Unix) | `\r\n` or `\n` |
| **Optional Header** | First row may contain column names | Header row followed by data rows |
| **Consistent Columns** | Each row must have the same number of fields | All rows match header count |

### Why RFC 4180 Matters for Data Engineering

**Without RFC 4180 compliance**, a CSV parser using simple string splitting will incorrectly parse any field containing the delimiter character. For example:

```
Input: "John","123 Main St, Apt 4","NYC"
split(','): ['"John"', '"123 Main St', ' Apt 4"', '"NYC"']  → 4 fields ❌
csv.reader: ['John', '123 Main St, Apt 4', 'NYC']            → 3 fields ✅
```

**With RFC 4180 compliance**, the parser understands that:
1. Double quotes define field boundaries, not just text decoration
2. Commas inside quoted fields are literal data, not structural separators
3. The field `"123 Main St, Apt 4"` is one value, not two

### Real-World Importance

RFC 4180 compliance is critical because:
- **Data Integrity**: Prevents address fields, descriptions, names, and free-text columns from being fragmented across multiple columns
- **Interoperability**: Ensures CSV files can be exchanged between systems (databases, spreadsheets, ETL tools) without corruption
- **Predictable Parsing**: Eliminates ambiguity when fields contain commas, quotes, or line breaks
- **Industry Standard**: Excel, Google Sheets, pandas, SQL databases, and most ETL tools all export RFC 4180-compliant CSV by default when fields contain special characters

### The Dual-Parser Approach

This notebook implements a dual-parser strategy that:
1. **Uses csv.reader (RFC 4180 compliant) as the primary parser** — correctly handles all standard CSV files
2. **Runs str.split() as a comparison parser** — detects when files contain quoted fields with embedded delimiters
3. **Flags parser disagreements** — surfaces exactly which rows have quoting that affects parsing
4. **Falls back gracefully** — uses split() when malformed quoting confuses the state-machine parser

This approach provides the correctness of RFC 4180 parsing with the resilience of a simple fallback, ensuring no data is lost regardless of CSV format quality.


# 3) Knowing and Confirming the Paths to the Data

In [1]:
import pandas as pd
import os
import glob

# Get all files in the Files section
files_path = "/lakehouse/default/Files"
all_files = glob.glob(f"{files_path}/**/*", recursive=True)

# Filter out directories (keep only actual files)
files_list = []
for file_path in all_files:
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        file_ext = os.path.splitext(file_name)[1] or "No extension"
        files_list.append({
            "file_name": file_name,
            "full_path": file_path,
            "size_mb": round(file_size / (1024 * 1024), 2),
            "extension": file_ext
        })

df_files = pd.DataFrame(files_list)
print(f"Total files in Files section: {len(df_files)}")
display(df_files)

# Store paths in variables for later use
for idx, row in df_files.iterrows():
    print(f"{row['file_name'].replace('.', '_').replace('-', '_')}_path = '{row['full_path']}'")

StatementMeta(, 8e41258a-8b56-4a06-b5f8-845d33ce9561, 3, Finished, Available, Finished, False)

Total files in Files section: 1


SynapseWidget(Synapse.DataFrame, 6a12923e-1d01-4ddf-a9d8-cf4d7f1b75b1)

AB_NYC_2019_csv_path = '/lakehouse/default/Files/AB_NYC_2019.csv'


# 4) Schema Validation Process

In [2]:
%pip install chardet

StatementMeta(, 8e41258a-8b56-4a06-b5f8-845d33ce9561, 8, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [1]:
"""
File Processing Pipeline for Microsoft Fabric
Uses Spark distributed text reading + Python CSV parsing on executors
Addresses: Memory Management, Quotation Detection, Header Parsing, Delta Table Naming
Includes: Full diagnostic logging, proper separator detection, encoding detection
"""

import re
import os
import csv
import io
import chardet
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Optional, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, lit, current_timestamp, monotonically_increasing_id

# ============================================================================
# CONFIGURATION
# ============================================================================

MAX_SAMPLE_SIZE = 100  # For profiling only
SAMPLE_BYTES = 8192    # Read first 8KB for encoding detection

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def sanitize_column_name(name: str) -> str:
    """Sanitize column names to be Delta-compliant with uniqueness guarantee."""
    if not name or not isinstance(name, str):
        return "_empty_column"
    
    name = re.sub(r'[ ,;{}()\n\t=]', '_', name.strip())
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    if name and name[0].isdigit():
        name = '_' + name
    if not name:
        name = '_empty'
    name = name.lower()
    
    return name

def format_table_name(file_name: str, suffix: str, bronze_schema: str) -> str:
    """Format Delta table name with proper naming conventions."""
    clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', file_name)
    clean_name = re.sub(r'_+', '_', clean_name)
    clean_name = clean_name.strip('_').lower()
    
    if clean_name and clean_name[0].isdigit():
        clean_name = 't_' + clean_name
    
    table_name = f"{bronze_schema}.{clean_name}_{suffix}"
    return table_name


def detect_file_properties(source_path: str) -> Dict[str, Any]:
    """
    Detect actual file properties: encoding, BOM, line endings, estimated rows.
    Returns actionable metadata used for processing decisions.
    """
    props = {
        'encoding': 'utf-8',
        'has_bom': False,
        'line_ending': '\n',
        'is_binary': False,
        'estimated_rows': 0,
        'warnings': []
    }
    
    try:
        # Read raw bytes for encoding detection
        with open(source_path, 'rb') as f:
            raw_bytes = f.read(SAMPLE_BYTES)
        
        if not raw_bytes:
            props['warnings'].append("File is empty")
            return props
        
        # Detect encoding using chardet
        detection = chardet.detect(raw_bytes)
        if detection and detection['encoding']:
            props['encoding'] = detection['encoding'].lower()
            if detection['confidence'] < 0.7:
                props['warnings'].append(f"Low confidence encoding detection ({detection['confidence']:.0%})")
        
        # Check for BOM
        if raw_bytes.startswith(b'\xef\xbb\xbf'):
            props['has_bom'] = True
            props['encoding'] = 'utf-8-sig'
        elif raw_bytes.startswith(b'\xff\xfe') or raw_bytes.startswith(b'\xfe\xff'):
            props['has_bom'] = True
            props['encoding'] = 'utf-16'
        
        # Detect line endings
        sample_text = raw_bytes.decode(props['encoding'], errors='replace')
        crlf_count = sample_text.count('\r\n')
        lf_count = sample_text.count('\n') - crlf_count
        cr_count = sample_text.count('\r') - crlf_count
        
        if crlf_count > lf_count and crlf_count > cr_count:
            props['line_ending'] = '\r\n'
        elif cr_count > lf_count:
            props['line_ending'] = '\r'
        else:
            props['line_ending'] = '\n'
        
        # Check if binary (null bytes or high concentration of non-printable chars)
        null_count = raw_bytes.count(b'\x00')
        if null_count > len(raw_bytes) * 0.1:
            props['is_binary'] = True
            props['warnings'].append("File appears to be binary, not text")
        
        # Estimate total rows from sample
        total_size = os.path.getsize(source_path)
        if total_size > 0 and len(sample_text) > 0:
            sample_lines = sample_text.count(props['line_ending'])
            if sample_lines > 0:
                avg_bytes_per_line = len(raw_bytes) / sample_lines
                props['estimated_rows'] = int(total_size / avg_bytes_per_line)
        
    except Exception as e:
        props['warnings'].append(f"Property detection failed: {str(e)[:100]}")
    
    return props

def detect_separator(source_path: str, file_props: Dict[str, Any], sample_lines: int = 15) -> str:
    """
    Robust separator detection with three-layer fallback.
    Layer 1: csv.Sniffer with full dialect detection
    Layer 2: Quote-aware consistency scoring with content validation
    Layer 3: Statistical analysis of field content patterns
    """
    candidates = [',', '|', '\t', ';']
    
    try:
        # Read sample with detected encoding
        with open(source_path, 'r', encoding=file_props['encoding']) as f:
            sample_lines_list = []
            for i, line in enumerate(f):
                if line.strip() and i <= sample_lines:
                    sample_lines_list.append(line.rstrip(file_props['line_ending']))
                if len(sample_lines_list) >= sample_lines:
                    break
        
        if not sample_lines_list:
            return ','
        
        sample_text = '\n'.join(sample_lines_list)
        
        # Layer 1: csv.Sniffer with full context
        try:
            sniffer = csv.Sniffer()
            # Sniffer looks at quoting, escaping, and delimiter patterns together
            dialect = sniffer.sniff(sample_text, delimiters=''.join(candidates))
            detected = dialect.delimiter
            
            # Validate Sniffer's choice with quote-aware parsing
            field_counts = []
            for line in sample_lines_list:
                try:
                    fields = len(next(csv.reader(io.StringIO(line), delimiter=detected)))
                    field_counts.append(fields)
                except:
                    continue
            
            if field_counts:
                most_common = max(set(field_counts), key=field_counts.count)
                consistency = field_counts.count(most_common) / len(field_counts)
                
                if consistency > 0.8 and most_common > 1:
                    return detected
                # Sniffer chose but validation failed, continue to Layer 2
        except:
            pass
        
        # Layer 2: Quote-aware consistency with content validation
        best_delimiter = ','
        best_score = -1
        
        for delim in candidates:
            field_counts = []
            all_fields = []
            
            for line in sample_lines_list:
                try:
                    fields = next(csv.reader(io.StringIO(line), delimiter=delim))
                    field_counts.append(len(fields))
                    all_fields.extend(fields)
                except:
                    continue
            
            if not field_counts:
                continue
            
            most_common_count = max(set(field_counts), key=field_counts.count)
            consistency_score = field_counts.count(most_common_count) / len(field_counts)
            
            # Content validation: check if fields look like real data
            non_empty_ratio = sum(1 for f in all_fields if f.strip()) / max(1, len(all_fields))
            
            # Length variation check: real data has varied field lengths
            field_lengths = [len(f) for f in all_fields if f.strip()]
            if field_lengths:
                unique_lengths = len(set(field_lengths))
                variation_score = min(1.0, unique_lengths / len(field_lengths)) if field_lengths else 0
            else:
                variation_score = 0
            
            # Composite score
            total_score = (consistency_score * 0.5) + (non_empty_ratio * 0.2) + (variation_score * 0.3)
            total_score *= most_common_count  # Prefer more columns (structured data)
            
            if total_score > best_score and most_common_count > 1:
                best_score = total_score
                best_delimiter = delim
        
        # Layer 3: If still uncertain, check if single-column
        if best_score <= 0:
            single_col_lines = 0
            for line in sample_lines_list:
                if len(line.split(',')) == 1 and len(line.split('\t')) == 1:
                    single_col_lines += 1
            if single_col_lines == len(sample_lines_list):
                return '\t'  # Default to tab for single column
        
        return best_delimiter
        
    except Exception as e:
        print(f"   ⚠️  Separator detection failed: {str(e)[:100]}")
        return ','

def detect_file_category(source_path: str, separator: str, file_props: Dict[str, Any], sample_size: int = 20) -> Tuple[str, Dict]:
    """Analyze the file to determine its quoting category."""
    try:
        with open(source_path, 'r', encoding=file_props['encoding']) as f:
            # Skip BOM if present
            if file_props['has_bom']:
                f.read(1)  # Consume BOM
            
            lines = []
            for i, line in enumerate(f):
                line = line.rstrip(file_props['line_ending']).strip()
                if line and i > 0:  # Skip header
                    lines.append(line)
                if len(lines) >= sample_size:
                    break
        
        if not lines:
            return "UNKNOWN", {}
        
        stats = {
            'total_fields': 0,
            'quoted_fields': 0,
            'unquoted_fields': 0,
            'lines_fully_quoted': 0,
            'lines_partially_quoted': 0,
            'lines_unquoted': 0,
            'lines_with_embedded_delimiters': 0
        }
        
        for line in lines:
            fields = next(csv.reader(io.StringIO(line), delimiter=separator))
            fields_split = line.split(separator)
            
            has_embedded_delimiters = (len(fields) != len(fields_split))
            if has_embedded_delimiters:
                stats['lines_with_embedded_delimiters'] += 1
            
            quoted_count = 0
            unquoted_count = 0
            
            for field in fields:
                if is_quoted_field(field, line):
                    quoted_count += 1
                else:
                    unquoted_count += 1
            
            stats['total_fields'] += len(fields)
            stats['quoted_fields'] += quoted_count
            stats['unquoted_fields'] += unquoted_count
            
            if quoted_count == len(fields):
                stats['lines_fully_quoted'] += 1
            elif quoted_count > 0:
                stats['lines_partially_quoted'] += 1
            else:
                stats['lines_unquoted'] += 1
        
        total_lines = len(lines)
        if stats['lines_fully_quoted'] == total_lines:
            category = "B - Fully Quoted"
        elif stats['lines_unquoted'] == total_lines and stats['lines_with_embedded_delimiters'] == 0:
            category = "A - Unquoted (Clean)"
        elif stats['lines_unquoted'] == total_lines and stats['lines_with_embedded_delimiters'] > 0:
            category = "D - Unquoted (Ambiguous)"
        elif stats['lines_partially_quoted'] > 0 or (stats['lines_fully_quoted'] > 0 and stats['lines_unquoted'] > 0):
            category = "C - Partially Quoted"
        else:
            category = "E - Mixed/Malformed"
        
        return category, stats
    except Exception as e:
        print(f"   ⚠️  Category detection failed: {str(e)[:100]}")
        return "UNKNOWN", {}

def is_quoted_field(field: str, raw_line: str, quote_char: str = '"') -> bool:
    """
    Properly detect if a field was quoted in the original line.
    Avoids false positives from substring matching.
    """
    escaped_field = re.escape(field)
    pattern = f'{quote_char}{escaped_field}{quote_char}'
    
    if re.search(pattern, raw_line):
        return True
    
    return False

def validate_header_from_file(source_path: str, separator: str, file_props: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    """Validate header using detected file properties."""
    warnings = []
    
    try:
        with open(source_path, 'r', encoding=file_props['encoding']) as f:
            # Handle BOM
            if file_props['has_bom']:
                f.read(1)
            
            first_line = f.readline().rstrip(file_props['line_ending']).strip()
            
            if not first_line:
                warnings.append("Empty header detected - file may have no data")
                return [], warnings
            
            raw_header = next(csv.reader(io.StringIO(first_line), delimiter=separator))
            
            if not raw_header:
                warnings.append("Empty header detected - no columns parsed")
                return [], warnings
            
            # Check for empty column names
            for i, col in enumerate(raw_header):
                if not col or not col.strip():
                    raw_header[i] = f"column_{i+1}"
                    warnings.append(f"Empty column name at position {i+1} renamed to '{raw_header[i]}'")
            
            # Sanitize all names
            sanitized = [sanitize_column_name(col) for col in raw_header]
            
            # Handle duplicates
            seen = {}
            final_header = []
            for i, name in enumerate(sanitized):
                if name in seen:
                    seen[name] += 1
                    new_name = f"{name}_{seen[name]}"
                    final_header.append(new_name)
                    warnings.append(f"Duplicate column '{name}' renamed to '{new_name}'")
                else:
                    seen[name] = 0
                    final_header.append(name)
            
            return final_header, warnings
            
    except Exception as e:
        warnings.append(f"Failed to read header: {str(e)[:100]}")
        return [], warnings

# ============================================================================
# SPARK DISTRIBUTED PROCESSOR (Fabric-Compatible)
# ============================================================================

def parse_line_worker(line_with_index: Tuple, separator: str, expected_columns: int) -> Tuple:
    """
    Worker function to parse a single CSV line with its line number.
    This runs on Spark executors in parallel.
    Input: (line_number, line_content)
    Returns (fields_list, error_info_or_None).
    """
    line_num, line = line_with_index
    
    try:
        fields_csv = next(csv.reader(io.StringIO(line), delimiter=separator))
    except Exception:
        return (None, (line_num, "PARSE_ERROR", "NO", 0, 0, expected_columns, line[:500]))
    
    if len(fields_csv) == expected_columns:
        return (fields_csv, None)
    else:
        fields_split = line.split(separator)
        if len(fields_split) == expected_columns:
            return (fields_split, None)
        else:
            error_type = "MISSING_COLUMNS" if len(fields_csv) < expected_columns else "EXTRA_COLUMNS"
            has_quotes = any(is_quoted_field(f, line) for f in fields_csv)
            error_info = (line_num, error_type, 'YES' if has_quotes else 'NO', len(fields_csv), len(fields_split), expected_columns, line[:500])
            return (None, error_info)


def process_file_spark(source_path: str, bronze_schema: str, spark: SparkSession) -> Tuple:
    """
    Process file using Spark to read text lines from local mount (Fabric compatible),
    then parse with Python's csv.reader on executors in parallel.
    """
    file_name = os.path.splitext(os.path.basename(source_path))[0]
    load_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    tables_created = []
    
    print(f"\n{'='*80}")
    print(f"📂 Processing: {file_name}")
    print(f"{'='*80}")
    
    # ========================================================================
    # DETECT FILE PROPERTIES
    # ========================================================================
    print(f"🔎 Analyzing file properties...")
    file_props = detect_file_properties(source_path)
    
    print(f"   📄 Encoding: {file_props['encoding']}")
    print(f"   📄 Line endings: {repr(file_props['line_ending'])}")
    if file_props['has_bom']:
        print(f"   📄 BOM detected: Yes (will be stripped)")
    if file_props['warnings']:
        for warning in file_props['warnings']:
            print(f"   ⚠️  Property Warning: {warning}")
    if file_props['is_binary']:
        print(f"   ❌ File appears to be binary - processing may fail")
    
    # ========================================================================
    # DETECT SEPARATOR
    # ========================================================================
    separator = detect_separator(source_path, file_props)
    
    separator_display = separator if separator != '\t' else '\\t'
    print(f"🔍 Detected separator: '{separator_display}'")
    
    # ========================================================================
    # DETECT FILE CATEGORY
    # ========================================================================
    file_category, category_stats = detect_file_category(source_path, separator, file_props)
    print(f"📋 File Category: {file_category}")
    if category_stats:
        print(f"   - Quoted fields: {category_stats.get('quoted_fields', 0)}")
        print(f"   - Unquoted fields: {category_stats.get('unquoted_fields', 0)}")
        print(f"   - Lines with embedded delimiters: {category_stats.get('lines_with_embedded_delimiters', 0)}")
        print(f"   - Fully quoted lines: {category_stats.get('lines_fully_quoted', 0)}")
        print(f"   - Partially quoted lines: {category_stats.get('lines_partially_quoted', 0)}")
        print(f"   - Unquoted lines: {category_stats.get('lines_unquoted', 0)}")
    
    # ========================================================================
    # VALIDATE HEADER
    # ========================================================================
    header, header_warnings = validate_header_from_file(source_path, separator, file_props)
    
    if header_warnings:
        for warning in header_warnings:
            print(f"   ⚠️  Header Warning: {warning}")
    
    if not header:
        print(f"   ❌ No valid header found")
        return file_name, 0, 0, 0, {}, [], 0, "ERROR", separator, "unknown"
    
    expected_column_count = len(header)
    print(f"   📋 Header: {expected_column_count} columns: {', '.join(header[:5])}{'...' if len(header) > 5 else ''}")
    
    # ========================================================================
    # PROCESS WITH SPARK
    # ========================================================================
    print(f"📥 Reading and processing with Spark distributed engine...")
    
    good_table = format_table_name(file_name, "consistent", bronze_schema)
    bad_table = format_table_name(file_name, "inconsistent", bronze_schema)
    
    try:
        local_path = f"file://{source_path}"
        text_rdd = spark.sparkContext.textFile(local_path)
        
        # Remove header and empty lines
        header_line = text_rdd.first()
        data_rdd = text_rdd.filter(lambda line: line != header_line and line.strip() != "")
        
        # Assign line numbers using zipWithIndex (0-based, so add 2 for 1-based + header offset)
        # Line numbers start at 2 because line 1 is the header
        indexed_rdd = data_rdd.zipWithIndex().map(lambda x: (x[1] + 2, x[0]))
        
        # Parse lines on executors with line numbers
        parsed_rdd = indexed_rdd.map(lambda x: parse_line_worker(x, separator, expected_column_count))
        parsed_rdd.cache()
        
        # Separate good and bad
        good_rdd = parsed_rdd.filter(lambda x: x[0] is not None).map(lambda x: x[0])
        bad_rdd = parsed_rdd.filter(lambda x: x[1] is not None).map(lambda x: x[1])
        
        # Add metadata to good records
        good_with_meta = good_rdd.map(lambda fields: fields + [load_timestamp, source_path, 'csv'])
        
        # Good DataFrame
        good_schema = StructType([
            StructField(col_name, StringType(), True) for col_name in header
        ] + [
            StructField('load_timestamp', StringType(), True),
            StructField('source_location', StringType(), True),
            StructField('file_type', StringType(), True)
        ])
        df_good = spark.createDataFrame(good_with_meta, good_schema)
        
        # Bad DataFrame - now includes line_number as first element
        bad_mapped = bad_rdd.map(lambda err: (
            err[0],  # line_number
            err[1],  # error_type
            err[2],  # parser_disagreement
            err[3],  # actual_fields_csv
            err[4],  # actual_fields_split
            err[5],  # expected_fields
            err[6],  # original_line
            load_timestamp,
            source_path,
            'csv'
        ))
        bad_schema = StructType([
            StructField('line_number', IntegerType(), True),
            StructField('error_type', StringType(), True),
            StructField('parser_disagreement', StringType(), True),
            StructField('actual_fields_csv', IntegerType(), True),
            StructField('actual_fields_split', IntegerType(), True),
            StructField('expected_fields', IntegerType(), True),
            StructField('original_line', StringType(), True),
            StructField('load_timestamp', StringType(), True),
            StructField('source_location', StringType(), True),
            StructField('file_type', StringType(), True)
        ])
        df_bad = spark.createDataFrame(bad_mapped, bad_schema)

        # Counts
        total_count = parsed_rdd.count()
        good_count = df_good.count()
        bad_count = total_count - good_count
        
        # Parser disagreements
        parser_disagreement_count = 0
        if bad_count > 0:
            parser_disagreement_count = df_bad.filter(col("parser_disagreement") == "YES").count()
        
        print(f"   📊 Parser Stats:")
        print(f"      - Spark distributed processing used across cluster")
        print(f"      - Parser disagreements detected: {parser_disagreement_count}")
        print(f"   ✅ Total: {total_count:,} | Good: {good_count:,} | Bad: {bad_count:,}")
        
        # Error breakdown
        error_types = {}
        quoted_file_detected = False
        
        if bad_count > 0:
            error_dist = df_bad.groupBy("error_type").count().collect()
            for row in error_dist:
                error_types[row["error_type"]] = row["count"]
            if parser_disagreement_count > 0:
                quoted_file_detected = True
        
        if quoted_file_detected:
            print(f"   🔍 QUOTED FILE DETECTED: Some rows have embedded delimiters in quoted fields")
        
        # Save to Delta
        print(f"💾 Saving to Bronze Delta tables...")
        
        if good_count > 0:
            df_good.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(good_table)
            tables_created.append(good_table)
            print(f"   ✅ Good records → {good_table} ({good_count:,} rows)")
        
        if bad_count > 0:
            df_bad.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable(bad_table)
            tables_created.append(bad_table)
            print(f"   ⚠️  Error records → {bad_table} ({bad_count:,} rows)")
        else:
            print(f"   ✅ No errors found - skipping error table creation")
        
        parsed_rdd.unpersist()
        
        return file_name, good_count, bad_count, total_count, error_types, tables_created, parser_disagreement_count, file_category, separator, 'csv'
        
    except Exception as e:
        error_msg = str(e)[:200]
        print(f"   ❌ Spark processing failed: {error_msg}")
        return file_name, 0, 0, 0, {}, [], 0, "ERROR", separator, "unknown"


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Main entry point."""
    
    bronze_schema = "Bronze"
    
    file_paths = [
       '/lakehouse/default/Files/AB_NYC_2019.csv'
    ]
    
    print("🚀 Starting PySpark Multi-File Processing Pipeline (Fabric-Optimized v2)...")
    print("="*80)
    print(f"📊 Total files to process: {len(file_paths)}")
    print(f"🗄️  Target schema: {bronze_schema}")
    print(f"🔧 Engine: Spark distributed text reading + Python CSV parsing on executors")
    print(f"🔍 Detection: Multi-layer separator detection + encoding/line ending detection")
    print(f"✅ Features: Proper Quote Detection | Header Validation | Clean Table Names | NO Memory Limits")
    print("="*80)
    
    spark = SparkSession.getActiveSession()
    if spark is None:
        raise Exception("❌ No active Spark session. Please attach a Lakehouse with Spark enabled.")
    
    print("✅ Spark session active")
    
    all_results = []
    total_good_all = 0
    total_bad_all = 0
    total_records_all = 0
    global_error_types = {}
    total_tables_created = 0
    total_parser_disagreements = 0
    clean_files = []
    dirty_files = []
    quoted_files = []
    file_categories = {}
    file_separators = {}
    
    for i, file_path in enumerate(file_paths, 1):
        print(f"\n{'#'*80}")
        print(f"# FILE {i} OF {len(file_paths)}")
        print(f"{'#'*80}")
        
        try:
            result = process_file_spark(file_path, bronze_schema, spark)
            all_results.append(result)
            
            (file_name, good_count, bad_count, total_count, error_types, 
             tables_created, parser_disagreements, file_category, separator, file_type) = result
            
            total_good_all += good_count
            total_bad_all += bad_count
            total_records_all += total_count
            total_tables_created += len(tables_created)
            total_parser_disagreements += parser_disagreements
            
            file_categories[file_name] = file_category
            file_separators[file_name] = separator
            
            if bad_count == 0 and total_count > 0:
                clean_files.append(file_name)
            else:
                dirty_files.append(file_name)
            
            if parser_disagreements > 0:
                quoted_files.append(file_name)
            
            for error_type, count in error_types.items():
                global_error_types[error_type] = global_error_types.get(error_type, 0) + count
                
        except Exception as e:
            print(f"❌ ERROR processing {os.path.basename(file_path)}: {str(e)[:200]}")
            all_results.append((os.path.basename(file_path), 0, 0, 0, {}, [], 0, "ERROR", ",", "unknown"))
    
    # Final Summary
    print("\n\n" + "="*80)
    print("✨ MULTI-FILE PIPELINE COMPLETE")
    print("="*80)
    print(f"📊 Files processed: {len(file_paths)}")
    print(f"   ✅ Clean files (no errors): {len(clean_files)}")
    print(f"   ⚠️  Dirty files (with errors): {len(dirty_files)}")
    print(f"   🔍 Files with parser disagreements: {len(quoted_files)}")
    print(f"📊 Total records across all files: {total_records_all:,}")
    print(f"✅ Total good records: {total_good_all:,}")
    print(f"❌ Total error records: {total_bad_all:,}")
    print(f"📊 Success rate: {(total_good_all/total_records_all*100):.2f}%" if total_records_all > 0 else "N/A")
    print(f"📂 Total Delta tables created: {total_tables_created}")
    print(f"🔍 Total parser disagreements: {total_parser_disagreements}")
    
    if quoted_files:
        print(f"\n🔍 FILES WITH PARSER DISAGREEMENTS:")
        for f in quoted_files:
            print(f"   • {f}")
    
    if file_separators:
        print(f"\n📋 SEPARATOR BREAKDOWN:")
        for fname in sorted(file_separators.keys()):
            sep_display = file_separators[fname] if file_separators[fname] != '\t' else '\\t'
            print(f"   • {fname}: '{sep_display}'")
    
    if file_categories:
        print(f"\n📋 FILE CATEGORY BREAKDOWN:")
        category_counts = {}
        for fname, cat in file_categories.items():
            category_counts[cat] = category_counts.get(cat, 0) + 1
        for cat, count in sorted(category_counts.items()):
            print(f"   • {cat}: {count} file(s)")
    
    if global_error_types:
        print(f"\n📉 Global Error Breakdown:")
        for error_type, count in sorted(global_error_types.items()):
            print(f"   • {error_type}: {count:,}")
    
    # Per-file summary
    print(f"\n{'='*80}")
    print(f"📋 PER-FILE SUMMARY")
    print(f"{'='*80}")
    print(f"{'File':<45} {'Total':>8} {'Good':>8} {'Bad':>8} {'Status':<8} {'Category':<28} {'Sep':<6} {'Tables'}")
    print(f"{'-'*45} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*28} {'-'*6} {'-'*6}")
    
    for result in all_results:
        (file_name, good_count, bad_count, total_count, _, tables_created, 
         parser_disagreements, file_category, separator, _) = result
        
        if bad_count == 0 and total_count > 0:
            status = "✅ CLEAN"
        elif total_count == 0:
            status = "❌ FAILED"
        else:
            status = "⚠️  DIRTY"
        
        num_tables = len(tables_created)
        cat_short = file_category[:28] if file_category else "UNKNOWN"
        sep_display = separator if separator != '\t' else '\\t'
        print(f"{file_name:<45} {total_count:>8,} {good_count:>8,} {bad_count:>8,} {status:<8} {cat_short:<28} {sep_display:<6} {num_tables} table(s)")

if __name__ == "__main__":
    main()

StatementMeta(, 5c8e0f23-19eb-43dd-9ab7-ab883f0562da, 3, Finished, Available, Finished, False)

🚀 Starting PySpark Multi-File Processing Pipeline (Fabric-Optimized v2)...
📊 Total files to process: 1
🗄️  Target schema: Bronze
🔧 Engine: Spark distributed text reading + Python CSV parsing on executors
🔍 Detection: Multi-layer separator detection + encoding/line ending detection
✅ Features: Proper Quote Detection | Header Validation | Clean Table Names | NO Memory Limits
✅ Spark session active

################################################################################
# FILE 1 OF 1
################################################################################

📂 Processing: AB_NYC_2019
🔎 Analyzing file properties...
   📄 Encoding: ascii
   📄 Line endings: '\n'
🔍 Detected separator: ','
📋 File Category: C - Partially Quoted
   - Quoted fields: 1
   - Unquoted fields: 319
   - Lines with embedded delimiters: 1
   - Fully quoted lines: 0
   - Partially quoted lines: 1
   - Unquoted lines: 19
   📋 Header: 16 columns: id, name, host_id, host_name, neighbourhood_group...
📥 Reading 